# Model Comparison & Evaluation (2-Phase)

**Run AFTER** all 3 models have been trained AND `Watershed.ipynb` has been run.

## What this notebook does

| Phase | Input | Metric |
|-------|-------|--------|
| Phase 1 — Raw | YOLOv8s-seg native masks, Mask R-CNN native masks, Faster R-CNN boxes (no masks) | Box mAP@50, Mask mAP@50, Counting MAE |
| Phase 2 — Watershed | All 3 models with watershed-refined masks | Same metrics — directly comparable |
| Delta | Phase 2 − Phase 1 | Does watershed improve each model? |

Files read:
- `runs/yolov8_seg/taco_v2/yolov8_seg_results.json`
- `runs/mask_rcnn/mask_rcnn_results.json`
- `runs/faster_rcnn/faster_rcnn_results.json`
- `runs/watershed_all_models/yolo_watershed_results.json`
- `runs/watershed_all_models/maskrcnn_watershed_results.json`
- `runs/watershed_all_models/fasterrcnn_watershed_results.json`

In [ ]:
import json, os, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import pandas as pd

BASE_DIR   = r"C:\Users\User\Desktop\Ipynb"
RUNS_DIR   = os.path.join(BASE_DIR, "runs")
OUTPUT_DIR = os.path.join(RUNS_DIR, "evaluation")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── File paths ─────────────────────────────────────────────────────────────────
RAW_PATHS = {
    "YOLOv8s-seg":  os.path.join(RUNS_DIR, "yolov8_seg", "taco_v2", "yolov8_seg_results.json"),
    "Mask R-CNN":   os.path.join(RUNS_DIR, "mask_rcnn",  "mask_rcnn_results.json"),
    "Faster R-CNN": os.path.join(RUNS_DIR, "faster_rcnn","faster_rcnn_results.json"),
}

WS_PATHS = {
    "YOLOv8s-seg":  os.path.join(RUNS_DIR, "watershed_all_models", "yolo_watershed_results.json"),
    "Mask R-CNN":   os.path.join(RUNS_DIR, "watershed_all_models", "maskrcnn_watershed_results.json"),
    "Faster R-CNN": os.path.join(RUNS_DIR, "watershed_all_models", "fasterrcnn_watershed_results.json"),
}

def load(paths):
    data = {}
    for name, path in paths.items():
        if os.path.exists(path):
            with open(path) as f:
                data[name] = json.load(f)
            print(f"  [ok] {name}")
        else:
            print(f"  [MISSING] {name}: {path}")
    return data

print("Loading raw results …")
raw = load(RAW_PATHS)
print("\nLoading watershed results …")
ws  = load(WS_PATHS)
print(f"\nRaw models loaded:       {list(raw.keys())}")
print(f"Watershed models loaded: {list(ws.keys())}")

## Phase 1 — Raw Model Results

YOLOv8s-seg: native instance segmentation masks  
Mask R-CNN: native mask-head output  
Faster R-CNN: bounding boxes only (mask_map50 = 0)

In [ ]:
KEYS = ["box_map50", "mask_map50", "counting_mae", "counting_within_1", "avg_inference_ms"]

def make_table(data_dict, label):
    rows = []
    for name, data in data_dict.items():
        m = data["metrics"]
        rows.append({
            "Model":          name,
            "Box mAP@50":     f"{m.get('box_map50', 0):.3f}",
            "Mask mAP@50":    f"{m.get('mask_map50', 0):.3f}",
            "Count MAE":      f"{m.get('counting_mae', 0):.2f}",
            "Within ±1 (%)":  f"{m.get('counting_within_1', 0):.1f}",
            "Within ±3 (%)":  f"{m.get('counting_within_3', 0):.1f}",
            "Inf (ms)":       f"{m.get('avg_inference_ms', 0):.0f}",
        })
    df = pd.DataFrame(rows).set_index("Model")
    print(f"\n{'='*65}")
    print(f"  {label}")
    print(f"{'='*65}")
    print(df.to_string())
    print(f"{'='*65}")
    return df

df_raw = make_table(raw, "PHASE 1 — RAW MODEL OUTPUTS")

## Phase 2 — After Watershed Post-Processing

All 3 models now produce segmentation masks (watershed-refined). Faster R-CNN gets mask capability for the first time.

In [ ]:
df_ws = make_table(ws, "PHASE 2 — AFTER WATERSHED")

## Delta Analysis — Phase 2 − Phase 1

Positive delta = watershed improved the metric.  
Negative delta = watershed hurt the metric (rare but possible for counting due to watershed under/over-segmentation).

In [ ]:
models_in_both = [m for m in raw if m in ws]

delta_rows = []
for name in models_in_both:
    r = raw[name]["metrics"]
    w = ws[name]["metrics"]
    delta_rows.append({
        "Model":             name,
        "ΔBox mAP@50":       w.get("box_map50", 0)    - r.get("box_map50", 0),
        "ΔMask mAP@50":      w.get("mask_map50", 0)   - r.get("mask_map50", 0),
        "ΔCount MAE":        w.get("counting_mae", 0) - r.get("counting_mae", 0),
        "ΔWithin±1 (pp)":    w.get("counting_within_1",0) - r.get("counting_within_1",0),
        "ΔInf (ms)":         w.get("avg_inference_ms",0)  - r.get("avg_inference_ms",0),
    })

df_delta = pd.DataFrame(delta_rows).set_index("Model")
print("\n" + "="*65)
print("  DELTA: Watershed vs Raw  (positive = improved)")
print("="*65)
print(df_delta.applymap(lambda x: f"{x:+.3f}" if isinstance(x, float) else x).to_string())
print("="*65)

# Delta bar chart
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
COLORS_DELTA = ["#4C72B0", "#55A868", "#C44E52"]

for ax, col in zip(axes, ["ΔBox mAP@50", "ΔMask mAP@50", "ΔCount MAE"]):
    vals   = df_delta[col].values
    colors = [("#2ecc71" if v >= 0 else "#e74c3c") for v in vals]
    bars   = ax.bar(df_delta.index, vals, color=colors)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(f"{col} (watershed − raw)", fontsize=11)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                v + (0.005 if v >= 0 else -0.01),
                f"{v:+.3f}", ha="center", fontsize=10)
    ax.tick_params(axis="x", rotation=12)
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("Effect of Watershed Post-Processing per Model", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "delta_watershed.png"), dpi=150)
plt.show()
print("Saved delta_watershed.png")

## Side-by-Side Comparison Charts

All 6 variants (3 raw + 3 watershed) plotted together for direct visual comparison.

In [ ]:
MODEL_NAMES = ["YOLOv8s-seg", "Mask R-CNN", "Faster R-CNN"]
PALETTE_RAW = ["#4C72B0", "#55A868", "#C44E52"]
PALETTE_WS  = ["#7aadda", "#8fd4a8", "#e88a7c"]

def get(data_dict, model, key, default=0.0):
    return data_dict.get(model, {}).get("metrics", {}).get(key, default)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

metrics_plot = [
    ("Box mAP@50",    "box_map50",         False, (0, 1)),
    ("Mask mAP@50",   "mask_map50",        False, (0, 1)),
    ("Count MAE",     "counting_mae",      True,  None),
    ("Count ±1 (%)",  "counting_within_1", False, (0, 100)),
    ("Count ±3 (%)",  "counting_within_3", False, (0, 100)),
    ("Inf time (ms)", "avg_inference_ms",  True,  None),
]

x = np.arange(len(MODEL_NAMES))
w = 0.38

for ax, (title, key, lower_better, ylim) in zip(axes.flat, metrics_plot):
    raw_vals = [get(raw, m, key) for m in MODEL_NAMES]
    ws_vals  = [get(ws,  m, key) for m in MODEL_NAMES]

    bars_r = ax.bar(x - w/2, raw_vals, w, label="Raw",       color=PALETTE_RAW)
    bars_w = ax.bar(x + w/2, ws_vals,  w, label="Watershed", color=PALETTE_WS,
                    hatch="//", edgecolor="grey")

    ax.set_title(title + (" (↓ better)" if lower_better else ""), fontsize=11)
    ax.set_xticks(x); ax.set_xticklabels(MODEL_NAMES, rotation=12, fontsize=9)
    if ylim:
        ax.set_ylim(*ylim)
    ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3)

    for bar in list(bars_r) + list(bars_w):
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + (ylim[1]*0.01 if ylim else h*0.02),
                f"{h:.2f}", ha="center", va="bottom", fontsize=7)

plt.suptitle("Raw vs Watershed: All 6 Model Variants", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "all_models_comparison.png"), dpi=150)
plt.show()
print("Saved all_models_comparison.png")

## Counting Error Distribution

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for row_idx, (phase_label, data_dict) in enumerate([("Raw", raw), ("Watershed", ws)]):
    for col_idx, model_name in enumerate(MODEL_NAMES):
        ax = axes[row_idx, col_idx]
        if model_name not in data_dict:
            ax.set_title(f"{model_name} [{phase_label}]\nNot available")
            ax.axis("off")
            continue
        errors = [r["count_error"] for r in data_dict[model_name].get("per_image", [])]
        if not errors:
            ax.set_title(f"{model_name} [{phase_label}]\nNo per_image data")
            ax.axis("off")
            continue
        color = (PALETTE_RAW if phase_label == "Raw" else PALETTE_WS)[col_idx]
        max_err = max(errors) if errors else 10
        ax.hist(errors, bins=range(0, max_err + 2), alpha=0.8, color=color, edgecolor="black")
        mae = np.mean(errors)
        ax.axvline(mae, color="red", linestyle="--", linewidth=1.5, label=f"MAE={mae:.1f}")
        ax.set_title(f"{model_name} [{phase_label}]", fontsize=11)
        ax.set_xlabel("Abs. Count Error"); ax.set_ylabel("Images")
        ax.legend(fontsize=9)

plt.suptitle("Counting Error Distribution (Raw vs Watershed)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "counting_error_distribution.png"), dpi=150)
plt.show()
print("Saved counting_error_distribution.png")

## Model Selection Scoring & Web App Recommendation

Scoring weights:
- **40%** mAP@50 (detection quality)
- **20%** mask mAP@50 (segmentation quality after watershed)
- **20%** counting accuracy within ±1
- **10%** inference speed
- **10%** model size

Uses the **watershed** results so the final model includes watershed as post-processing.

In [ ]:
MODEL_SIZES = {"YOLOv8s-seg": 23.0, "Mask R-CNN": 170.0, "Faster R-CNN": 170.0}

def normalize(vals, lower_is_better=False):
    v = np.array(vals, dtype=float)
    if v.max() == v.min():
        return np.ones_like(v) * 0.5
    n = (v - v.min()) / (v.max() - v.min())
    return 1 - n if lower_is_better else n

avail = [m for m in MODEL_NAMES if m in ws]
map50  = [get(ws, m, "box_map50") for m in avail]
mmask  = [get(ws, m, "mask_map50") for m in avail]
cacc   = [get(ws, m, "counting_within_1") / 100 for m in avail]
speed  = [get(ws, m, "avg_inference_ms") for m in avail]
sizes  = [MODEL_SIZES[m] for m in avail]

n_map50 = normalize(map50)
n_mask  = normalize(mmask)
n_cacc  = normalize(cacc)
n_speed = normalize(speed, lower_is_better=True)
n_size  = normalize(sizes, lower_is_better=True)

W = dict(map50=0.40, mask=0.20, count=0.20, speed=0.10, size=0.10)
scores = {}
print("="*70)
print("MODEL SELECTION SCORING  (watershed variants)")
print("="*70)
print(f"{'Model':<16} {'mAP50':>7} {'Mask':>7} {'Count':>7} {'Speed':>7} {'Size':>7} {'TOTAL':>7}")
print("-"*70)
for i, name in enumerate(avail):
    s = (W["map50"]*n_map50[i] + W["mask"]*n_mask[i] + W["count"]*n_cacc[i]
         + W["speed"]*n_speed[i] + W["size"]*n_size[i])
    scores[name] = s
    print(f"{name:<16} {n_map50[i]:>7.3f} {n_mask[i]:>7.3f} {n_cacc[i]:>7.3f}"
          f" {n_speed[i]:>7.3f} {n_size[i]:>7.3f} {s:>7.3f}")
print("="*70)

best = max(scores, key=scores.get)
print(f"\nRECOMMENDED FOR WEB APP: {best}  (score={scores[best]:.3f})")
print("  Deploy as: {model} + Watershed post-processing on each bounding box")

# Radar chart
N = 5
angles = [n / N * 2 * np.pi for n in range(N)]
angles += angles[:1]
labels = ["mAP@50", "Mask mAP", "Counting\nAccuracy", "Speed", "Size"]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
for i, name in enumerate(avail):
    vals = [n_map50[i], n_mask[i], n_cacc[i], n_speed[i], n_size[i]]
    vals += vals[:1]
    c = PALETTE_WS[MODEL_NAMES.index(name)]
    ax.plot(angles, vals, linewidth=2, label=name, color=c)
    ax.fill(angles, vals, alpha=0.1, color=c)

ax.set_xticks(angles[:-1]); ax.set_xticklabels(labels, fontsize=11)
ax.set_ylim(0, 1.15)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), fontsize=11)
ax.set_title("Model Scoring (Watershed variants)", fontsize=13, fontweight="bold", pad=20)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "radar_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved radar_comparison.png")

# Save final report
report = {
    "recommendation": best,
    "scores": scores,
    "phase1_raw":       {m: raw[m]["metrics"] for m in raw},
    "phase2_watershed": {m: ws[m]["metrics"]  for m in ws},
    "delta": {name: {
        "Δbox_map50":    get(ws,name,"box_map50")    - get(raw,name,"box_map50"),
        "Δmask_map50":   get(ws,name,"mask_map50")   - get(raw,name,"mask_map50"),
        "Δcounting_mae": get(ws,name,"counting_mae") - get(raw,name,"counting_mae"),
    } for name in models_in_both},
}
with open(os.path.join(OUTPUT_DIR, "final_report.json"), "w") as f:
    json.dump(report, f, indent=2)
print(f"\nFinal report saved: {OUTPUT_DIR}/final_report.json")
print("\n" + "="*70)
print(f"FINAL RECOMMENDATION:  {best} + Watershed post-processing")
print("="*70)